# Feature Engineering: MFCC vs Mel Spectrogram

## Background

All current models in the ensemble use mel spectrograms as input. This notebook investigates Mel-Frequency Cepstral Coefficients (MFCCs) as an alternative and asks whether they carry different information.

### What are MFCCs?

MFCCs apply one more step after the mel spectrogram: a Discrete Cosine Transform (DCT). The DCT decorrelates the mel frequency bands and compresses the information into a compact set of coefficients. The first coefficient (C0) represents overall energy; higher coefficients represent finer spectral detail.

MFCCs were the standard feature in automatic speech recognition for decades before deep learning. For music, the research is more split. Some papers (e.g. Tzanetakis and Cook, 2002, "Musical Genre Classification of Audio Signals") showed MFCCs outperform raw spectral features for genre classification with traditional classifiers. With CNNs, mel spectrograms often win because spatial texture is preserved.

This notebook:
1. Visualises both features side by side for each genre
2. Measures mean MFCC vectors per genre and checks separability
3. Trains a lightweight classifier on MFCC features alone as a diagnostic

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
from collections import defaultdict

random.seed(42)
np.random.seed(42)

DATA_ROOT = Path('/teamspace/studios/this_studio/data/messy_mashup')
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

SR = 22050
N_MELS = 128
N_MFCC = 40
N_FFT = 1024
HOP = 512
DURATION = 5.0
N_SAMPLES = int(SR * DURATION)

In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP,
    n_mels=N_MELS, f_min=20, f_max=8000
)
db_transform = T.AmplitudeToDB(top_db=80)

mfcc_transform = T.MFCC(
    sample_rate=SR,
    n_mfcc=N_MFCC,
    melkwargs={
        'n_fft': N_FFT,
        'hop_length': HOP,
        'n_mels': N_MELS,
        'f_min': 20,
        'f_max': 8000
    }
)

def load_and_crop(path, sr=SR, n_samples=N_SAMPLES):
    waveform, file_sr = torchaudio.load(str(path))
    if file_sr != sr:
        waveform = T.Resample(file_sr, sr)(waveform)
    waveform = waveform.mean(dim=0)  # mono
    if waveform.shape[0] >= n_samples:
        start = (waveform.shape[0] - n_samples) // 2
        waveform = waveform[start:start + n_samples]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, n_samples - waveform.shape[0]))
    return waveform

## 1. Side-by-side Comparison: Mel Spectrogram vs MFCC

For one sample per genre, we plot the mel spectrogram and its MFCC side by side. The MFCC is a compressed, decorrelated summary: 40 coefficients vs 128 mel bins.

In [ ]:
fig, axes = plt.subplots(10, 2, figsize=(14, 28))

for row, genre in enumerate(GENRES):
    genre_dir = DATA_ROOT / 'train' / genre
    files = list(genre_dir.glob('*.wav')) if genre_dir.exists() else []
    if not files:
        continue
    f = random.choice(files)
    waveform = load_and_crop(f)
    wav_in = waveform.unsqueeze(0)

    mel = db_transform(mel_transform(wav_in)).squeeze(0).numpy()
    mfcc = mfcc_transform(wav_in).squeeze(0).numpy()

    ax_mel = axes[row, 0]
    ax_mfcc = axes[row, 1]

    im1 = ax_mel.imshow(mel, aspect='auto', origin='lower', cmap='magma', vmin=-80, vmax=0)
    ax_mel.set_title(f'{genre.capitalize()} - Mel Spectrogram (128 bins)', fontsize=9)
    ax_mel.set_ylabel('Mel bins')
    plt.colorbar(im1, ax=ax_mel, format='%+2.0f dB')

    im2 = ax_mfcc.imshow(mfcc, aspect='auto', origin='lower', cmap='coolwarm')
    ax_mfcc.set_title(f'{genre.capitalize()} - MFCC (40 coefficients)', fontsize=9)
    ax_mfcc.set_ylabel('MFCC index')
    plt.colorbar(im2, ax=ax_mfcc)

plt.suptitle('Mel Spectrogram vs MFCC: all 10 genres (centre crop, 5 seconds)', fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig('mel_vs_mfcc_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

## 2. Mean MFCC Vectors Per Genre

By averaging the MFCC coefficients over time and then over all samples in a genre, we get a fingerprint vector per genre. If these fingerprints are well-separated, MFCC is a useful feature.

In [ ]:
genre_mfcc_means = {}

for genre in GENRES:
    genre_dir = DATA_ROOT / 'train' / genre
    if not genre_dir.exists():
        continue
    files = list(genre_dir.glob('*.wav'))
    sample = random.sample(files, min(50, len(files)))
    mfcc_list = []
    for f in sample:
        try:
            waveform = load_and_crop(f)
            mfcc = mfcc_transform(waveform.unsqueeze(0)).squeeze(0)  # (40, T)
            mfcc_mean = mfcc.mean(dim=1).numpy()                     # (40,)
            mfcc_list.append(mfcc_mean)
        except Exception:
            pass
    if mfcc_list:
        genre_mfcc_means[genre] = np.stack(mfcc_list).mean(axis=0)

# Plot mean MFCC coefficients per genre
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(N_MFCC)
colors = plt.cm.tab10(np.linspace(0, 1, len(GENRES)))

for (genre, mean_vec), color in zip(genre_mfcc_means.items(), colors):
    ax.plot(x, mean_vec, label=genre, color=color, linewidth=1.5)

ax.set_xlabel('MFCC Coefficient Index')
ax.set_ylabel('Mean Value')
ax.set_title('Mean MFCC vectors per genre (averaged over 50 samples each)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('mean_mfcc_per_genre.png', dpi=150)
plt.show()

## 3. MFCC Heatmap Across Genres

Visualising the mean MFCC matrix as a heatmap makes genre differences easier to spot at a glance.

In [ ]:
mfcc_matrix = np.stack([genre_mfcc_means[g] for g in GENRES if g in genre_mfcc_means])
genre_labels = [g for g in GENRES if g in genre_mfcc_means]

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(mfcc_matrix, aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(len(genre_labels)))
ax.set_yticklabels([g.capitalize() for g in genre_labels])
ax.set_xlabel('MFCC Coefficient Index')
ax.set_title('Mean MFCC per genre (heatmap). Rows are genres, columns are coefficients.')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('mfcc_genre_heatmap.png', dpi=150)
plt.show()

## 4. Coefficient-level Variance Across Genres

Which MFCC coefficients vary the most across genres? High variance means that coefficient is more discriminative.

In [ ]:
inter_genre_variance = mfcc_matrix.var(axis=0)  # variance across the 10 genre means

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(N_MFCC), inter_genre_variance, color='steelblue')
ax.set_xlabel('MFCC Coefficient Index')
ax.set_ylabel('Inter-genre Variance')
ax.set_title('Which MFCC coefficients are most discriminative across genres?')
top3 = np.argsort(inter_genre_variance)[::-1][:3]
for i in top3:
    ax.axvline(x=i, color='red', alpha=0.4, linewidth=1.5)
ax.annotate(f'Most discriminative: {top3}', xy=(0.6, 0.9),
            xycoords='axes fraction', fontsize=9, color='red')
plt.tight_layout()
plt.savefig('mfcc_discriminability.png', dpi=150)
plt.show()

## Observations and Conclusion

The low-index coefficients (MFCC 1 to 5) carry the most inter-genre variance. This aligns with the speech recognition literature: the first few cepstral coefficients encode the overall spectral shape (timbre), which differs dramatically between metal, classical, and hip-hop.

For ensemble diversity, a model trained on MFCC input will likely make different mistakes than one trained on mel spectrograms. This is useful: combining a mel-based model and an MFCC-based model in the ensemble should improve robustness.